In [ ]:
#@title 1. Configuración del Entorno
import os

# Clonar el repositorio si no existe
if not os.path.exists("Signal-Analysis-Vol-I"):
    !git clone https://github.com/ecundir/Signal-Analysis-Vol-I.git

# Cambiar a la carpeta del capítulo
%cd Signal-Analysis-Vol-I/capitulo_8
print("✅ Entorno configurado correctamente.")

**Control de Posición Vertical de un UAV(Unmanned Aerial Vehicle) o Drone**

Este laboratorio invita a actuar como arquitecto de sistemas, diseñando el control de descenso de un UAV mediante la ubicación estratégica de polos en el plano complejo s para cumplir con un contrato de precisión industrial. Para operar el simulador en Google Colab, solo debes desplazar los controles deslizantes (widgets) de amortiguamiento (ζ) y frecuencia natural (ωn​); la plataforma recalculará instantáneamente la respuesta temporal y el mapa de polos, permitiendo identificar visualmente cómo el ángulo y la distancia de las raíces al origen dictan si el dron aterriza con suavidad o si el impacto excede los límites de seguridad estructural.

In [ ]:
# =============================================================================
# LABORATORIO VIVO: SÍNTESIS DE CONTROL PARA ATERRIZAJE DE UAV
# =============================================================================

# 1. PREPARACIÓN DEL ENTORNO
# -----------------------------------------------------------------------------
!pip install control -q

import numpy as np
import matplotlib.pyplot as plt
import control as ctrl
import ipywidgets as widgets
from IPython.display import display, HTML

# 2. DEFINICIÓN DEL MOTOR DE SIMULACIÓN TÉCNICA
# -----------------------------------------------------------------------------
def sintetizar_uav(zeta, wn):
    """
    Simula la dinámica de descenso de un dron basado en la ubicación de polos.
    Calcula métricas de desempeño y valida contra restricciones del cliente.
    """
    # Modelo de Segundo Orden Estándar: G(s) = wn^2 / (s^2 + 2*zeta*wn*s + wn^2)
    num = [wn**2]
    den = [1, 2*zeta*wn, wn**2]
    H = ctrl.TransferFunction(num, den)

    # Simulación de respuesta al escalón (Referencia: Descenso de 10m a 0m)
    t = np.linspace(0, 5, 1000)
    t, y = ctrl.step_response(H, t)

    # Transformación: La respuesta al escalón unitario (0 a 1) se escala
    # para representar un descenso controlado (10m a 0m)
    altura = 10 * (1 - y)

    # Extracción de Métricas de Desempeño
    info = ctrl.step_info(H)
    Mp = info['Overshoot']
    Ts = info['SettlingTime']

    # --- VISUALIZACIÓN ACADÉMICA ---
    fig = plt.figure(figsize=(14, 6))

    # A. Gráfico de Respuesta Temporal (Dominio del Tiempo)
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.plot(t, altura, 'k-', lw=2, label='Trayectoria del UAV')
    ax1.axhline(0, color='red', lw=1.5, label='Suelo (Objetivo)')

    # Zona de tolerancia (Banda de error de 5cm para el aterrizaje)
    ax1.fill_between(t, -0.05, 0.05, color='green', alpha=0.2, label='Zona de Seguridad (5cm)')
    ax1.axvline(3.0, color='orange', ls='--', label='Límite Contrato (3s)')

    ax1.set_title('Respuesta Temporal: Descenso de Precisión', fontsize=12)
    ax1.set_xlabel('Tiempo [s]')
    ax1.set_ylabel('Altitud [m]')
    ax1.set_ylim(-0.5, 11)
    ax1.grid(True, which='both', linestyle=':', alpha=0.6)
    ax1.legend(loc='upper right')

    # B. Mapa de Polos y Ceros (Plano s)
    ax2 = fig.add_subplot(1, 2, 2)
    poles = ctrl.poles(H)
    ax2.scatter(poles.real, poles.imag, marker='x', color='blue', s=100, label='Polos de Síntesis')

    # Dibujar la envolvente de diseño (Frontera Ts < 3s -> sigma > 4/3)
    ax2.axvline(-1.33, color='orange', ls='--', alpha=0.5)
    ax2.fill_betweenx([-15, 15], -1.33, 0, color='red', alpha=0.05, label='Región Lenta')

    ax2.axvline(0, color='black', lw=1)
    ax2.axhline(0, color='black', lw=1)
    ax2.set_title(r'Configuración en el Plano $s$', fontsize=12)
    ax2.set_xlabel(r'$\sigma$ (Atenuación)')
    ax2.set_ylabel(r'$j\omega$ (Frecuencia Amortiguada)')
    ax2.set_xlim(-8, 1)
    ax2.set_ylim(-8, 8)
    ax2.grid(True, which='both', linestyle=':', alpha=0.6)
    ax2.legend()

    plt.tight_layout()
    plt.show()

   # 3. MÓDULO DE EVALUACIÓN DE CALIDAD (SÍ/NO) CON CORRECCIÓN DE ESCAPE
    # -------------------------------------------------------------------------
    cumple_mp = Mp <= 0.5
    cumple_ts = Ts <= 3.0
    resultado = "APROBADO" if cumple_mp and cumple_ts else "RECHAZADO"
    color = "#28a745" if resultado == "APROBADO" else "#dc3545"

    # Usamos r''' para que las barras invertidas de LaTeX y HTML no causen Warning
    html_report = r"""
    <div style="font-family: sans-serif; border-left: 10px solid {color}; padding: 15px; background: #f8f9fa;">
        <h3 style="color: {color}; margin-top: 0;">Reporte de Control de Calidad: {resultado}</h3>
        <table style="width: 100%; border-collapse: collapse;">
            <tr>
                <td style="padding: 5px;"><b>Sobreimpulso (Mp):</b> {Mp:.3f}%</td>
                <td style="padding: 5px;">Requisito: ≤ 0.5%</td>
                <td style="padding: 5px;">{icon_mp}</td>
            </tr>
            <tr>
                <td style="padding: 5px;"><b>Tiempo de Asentamiento (Ts):</b> {Ts:.3f} s</td>
                <td style="padding: 5px;">Requisito: < 3.0 s</td>
                <td style="padding: 5px;">{icon_ts}</td>
            </tr>
        </table>
        <p style="font-size: 0.9em; margin-bottom: 0;">
            <b>Coordenadas de Síntesis:</b> $s = {sigma:.2f} \pm j{wd:.2f}$
        </p>
    </div>
    """.format(
        color=color,
        resultado=resultado,
        Mp=Mp,
        Ts=Ts,
        icon_mp='✅' if cumple_mp else '❌',
        icon_ts='✅' if cumple_ts else '❌',
        sigma=-zeta*wn,
        wd=wn*np.sqrt(max(0, 1-zeta**2))
    )
    display(HTML(html_report))

# 4. INTERFAZ DE SÍNTESIS INTERACTIVA
# -----------------------------------------------------------------------------
ui = widgets.interact(
    sintetizar_uav,
    zeta = widgets.FloatSlider(
        min=0.4, max=1.1, step=0.01, value=0.6,
        description=r'Amort. ($\zeta$)', readout_format='.2f'
    ),
    wn = widgets.FloatSlider(
        min=1.0, max=8.0, step=0.1, value=2.0,
        description=r'Frec. ($\omega_n$)', readout_format='.1f'
    )
)